# 🔗 05 - Relationships and Correlation

Notebook 04 compared a numeric column across **groups**. This notebook compares two **numeric** columns with each other: when one goes up, does the other tend to go up, go down, or do nothing?

The tool is **correlation**: one number from −1 to 1. It's useful, it's everywhere, and it's easy to over-trust. This notebook ends with four datasets that share the same correlation and look nothing alike, which is the reason Module 08 exists.

## ✅ Learning Goals
- Describe a relationship by its **direction** and **strength**
- Compute Pearson correlation with `.corr()`, for one pair or a whole table
- Find the strongest pairs in a correlation matrix
- Choose **Spearman** correlation for ranked, curved, or outlier-heavy data
- Explain why r ≈ 0 doesn't mean "no relationship," and why correlation isn't causation

In [1]:
import numpy as np
import pandas as pd

poke = pd.read_csv('pokemon.csv')
titanic = pd.read_csv('titanic.csv')

stats = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

## 1. Direction and Strength
| Relationship | Meaning | Example |
|---|---|---|
| **Positive** | as one rises, the other tends to rise | study hours and exam score |
| **Negative** | as one rises, the other tends to fall | car age and resale price |
| **None** | knowing one tells you nothing about the other | shoe size and exam score |

"Tends to" is doing a lot of work. A positive relationship means **on average**, not for every single row.

### Covariance: The Right Idea, Awkward Units
**Covariance** measures whether two columns move together. It's positive when they rise together and negative when one rises as the other falls:

In [2]:
print('Covariance of Attack and Defense:', round(poke['Attack'].cov(poke['Defense']), 1))
print('Covariance of Attack/10 and Defense:', round((poke['Attack'] / 10).cov(poke['Defense']), 1))

Covariance of Attack and Defense: 444.0
Covariance of Attack/10 and Defense: 44.4


Dividing Attack by 10 changed the covariance by 10×, even though the relationship didn't change at all. Covariance depends on the **units**, so its size is hard to interpret. Correlation fixes that.

## 2. Pearson Correlation
**Pearson's r** is covariance rescaled to always land between −1 and 1, with no units:

In [3]:
print('r(Attack, Defense):    ', round(poke['Attack'].corr(poke['Defense']), 3))
print('r(Attack/10, Defense): ', round((poke['Attack'] / 10).corr(poke['Defense']), 3))

r(Attack, Defense):     0.439
r(Attack/10, Defense):  0.439


Same answer either way. A rough guide for reading |r|:

| \|r\| | Strength |
|---|---|
| below 0.1 | negligible |
| 0.1 – 0.3 | weak |
| 0.3 – 0.5 | moderate |
| 0.5 – 0.7 | strong |
| above 0.7 | very strong |

The sign gives the direction. Pearson's r measures **straight-line** (linear) relationships only. Hold on to that for section 5.

## 3. A Whole Correlation Matrix
`df.corr()` computes r for **every pair** of columns at once:

In [4]:
corr = poke[stats].corr()
corr.round(2)

,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed
HP,1.00,0.42,0.24,0.36,0.38,0.18
Attack,0.42,1.00,0.44,0.40,0.26,0.38
Defense,0.24,0.44,1.00,0.22,0.51,0.02
Sp. Atk,0.36,0.40,0.22,1.00,0.51,0.47
Sp. Def,0.38,0.26,0.51,0.51,1.00,0.26
Speed,0.18,0.38,0.02,0.47,0.26,1.00


How to read it:
- The diagonal is always 1 (every column correlates perfectly with itself).
- The table is **symmetric**: row `Attack`, column `Defense` equals row `Defense`, column `Attack`.
- So only the cells **above** the diagonal hold new information.

### Finding the Strongest Pairs
Keep the upper triangle, then reshape the table into one row per pair with `.stack()`:

In [5]:
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

pairs = upper.stack().dropna().rename('r').sort_values(key=abs, ascending=False)
pairs.round(3).head(5)

Defense  Sp. Def    0.511
Sp. Atk  Sp. Def    0.506
         Speed      0.473
Attack   Defense    0.439
HP       Attack     0.422
Name: r, dtype: float64

`np.triu(..., k=1)` marks the cells strictly above the diagonal. `.where()` keeps those and turns the rest into `NaN`. `.stack()` turns the table into one row per (row, column) pair, and `.dropna()` throws away the `NaN` pairs. Sorting with `key=abs` ranks by strength, regardless of sign.

In Module 08 this matrix becomes a heatmap, which makes the pattern visible at a glance.

✅ **Your Turn:** Which stat has the **weakest** relationship with `Speed`? (Hint: `corr['Speed']`, then sort.)

In [7]:
# Your turn here
corr['Speed'].sort_values(key=abs, ascending=False)

Speed      1.000000
Sp. Atk    0.473018
Attack     0.381240
Sp. Def    0.259133
HP         0.175952
Defense    0.015227
Name: Speed, dtype: float64

In [8]:
weakest = corr['Speed'].drop('Speed').sort_values(key=abs).head(1)
print(weakest)

Defense    0.015227
Name: Speed, dtype: float64


## 4. Spearman Correlation: When Order Is What Matters
**Spearman's ρ** (rho) replaces each value with its **rank** (1st, 2nd, 3rd, …) and computes Pearson's r on the ranks. That makes it a good choice when:
1. the data are **ordinal** (notebook 01): ranks are all an ordinal column really has
2. the relationship is **curved but always rising (or always falling)**
3. there are **outliers**: a rank can only be "last," no matter how extreme the value

### Curved but always rising

In [9]:
x = pd.Series(range(1, 21))
y = x ** 3

print('Pearson: ', round(x.corr(y), 3))
print('Spearman:', round(x.corr(y, method='spearman'), 3))

Pearson:  0.922
Spearman: 1.0


`y` rises **every single time** `x` rises, so the ranks line up perfectly (Spearman = 1). Pearson is lower because the points don't lie on a straight line.

### Outliers
One extreme point can create (or destroy) a Pearson correlation:

In [10]:
np.random.seed(21)
a = pd.Series(np.random.normal(50, 10, 30))
b = pd.Series(np.random.normal(50, 10, 30))      # unrelated to a

print('No outlier:   Pearson', round(a.corr(b), 3), '| Spearman', round(a.corr(b, method='spearman'), 3))

a_out = pd.concat([a, pd.Series([300])], ignore_index=True)
b_out = pd.concat([b, pd.Series([300])], ignore_index=True)
print('One outlier:  Pearson', round(a_out.corr(b_out), 3), '| Spearman', round(a_out.corr(b_out, method='spearman'), 3))

No outlier:   Pearson 0.002 | Spearman -0.05
One outlier:  Pearson 0.959 | Spearman 0.049


A single point turned two **unrelated** columns into a "very strong" Pearson correlation. Spearman barely noticed. Notebook 03's lesson applies here too: check for outliers before you trust a Pearson r.

### Ordinal data: Titanic class and fare
`Pclass` is ordinal and `Fare` is very skewed, so Spearman is the better fit:

In [11]:
print('Pearson: ', round(titanic['Pclass'].corr(titanic['Fare']), 3))
print('Spearman:', round(titanic['Pclass'].corr(titanic['Fare'], method='spearman'), 3))

Pearson:  -0.549
Spearman: -0.688


Both are negative (a higher class **number**, meaning a cheaper class, goes with a lower fare). Spearman shows the relationship is stronger than Pearson suggests, once the skew in `Fare` stops distorting the calculation.

| Use | When |
|---|---|
| **Pearson** (`method='pearson'`, the default) | both columns are interval/ratio, the pattern is roughly a straight line, and there are no wild outliers |
| **Spearman** (`method='spearman'`) | ordinal data, curved-but-monotonic patterns, or outliers/skew |

## 5. r ≈ 0 Does Not Mean "No Relationship"
Both methods measure relationships that go **one way**. A relationship that rises and then falls can score zero:

In [12]:
x = pd.Series(np.linspace(-10, 10, 41))
y = x ** 2          # y is completely determined by x

print('Pearson: ', round(x.corr(y), 3))
print('Spearman:', round(x.corr(y, method='spearman'), 3))

Pearson:  0.0
Spearman: 0.0


`y` is a perfect function of `x`, and both correlations are 0. Correlation can tell you a relationship **exists**. It can't promise that one **doesn't**.

## 6. Correlation Is Not Causation
Titanic `Fare` and `Survived` are positively correlated:

In [13]:
print('Overall r(Fare, Survived):', round(titanic['Fare'].corr(titanic['Survived']), 3))

Overall r(Fare, Survived): 0.257


Did buying an expensive ticket *cause* survival? Notebook 04 showed that **class** drives both: 1st class paid more **and** survived more. Check the correlation *within* each class:

In [ ]:
for cls in [1, 2, 3]:
    in_class = titanic[titanic['Pclass'] == cls]
    r = in_class['Fare'].corr(in_class['Survived'])
    print(f'Class {cls}: r = {r:.3f}  (n = {len(in_class)})')

Class 1: r = 0.191  (n = 216)
Class 2: r = 0.099  (n = 184)
Class 3: r = 0.001  (n = 491)


Within classes, the relationship is weaker. Much of the overall correlation came from the **confounder** (class), not from fare itself. A correlation can come from:
- **A causes B**, or **B causes A** (reverse causation)
- **C causes both** (confounding, like class here)
- **Coincidence**, especially when you test many pairs and report the strongest one

A correlation is a reason to ask **why**. It isn't the answer.

## 7. Same Numbers, Different Data: Anscombe's Quartet
`anscombe.csv` holds four small datasets, `I` through `IV`, each with 11 `(x, y)` pairs. Summarize each one with everything from this module:

In [15]:
anscombe = pd.read_csv('anscombe.csv')

summary = anscombe.groupby('dataset').agg(
    x_mean=('x', 'mean'), x_std=('x', 'std'),
    y_mean=('y', 'mean'), y_std=('y', 'std'),
)
summary['r'] = anscombe.groupby('dataset').apply(lambda d: d['x'].corr(d['y']))
summary.round(2)

,x_mean,x_std,y_mean,y_std,r
dataset,,,,,
I,9.0,3.32,7.5,2.03,0.82
II,9.0,3.32,7.5,2.03,0.82
III,9.0,3.32,7.5,2.03,0.82
IV,9.0,3.32,7.5,2.03,0.82


The four datasets are **statistically identical**: the same means, the same standard deviations, and the same correlation (0.82, "very strong"). By the numbers, they're the same dataset four times.

They aren't. One is a noisy straight line. One is a perfect curve. One is a perfect line with a single outlier. One is a vertical stack of points, with one far-off point creating the entire correlation. You can see some of it by printing the raw values, but that doesn't scale past 11 rows:

In [16]:
anscombe[anscombe['dataset'] == 'IV']

,dataset,x,y
33,IV,8,6.58
34,IV,8,5.76
35,IV,8,7.71
36,IV,8,8.84
37,IV,8,8.47
38,IV,8,7.04
39,IV,8,5.25
40,IV,19,12.50
41,IV,8,5.56
42,IV,8,7.91


> 🎯 **Summary statistics compress data, and compression throws information away.** Notebooks 01–05 gave you the numbers. Module 08 opens by plotting these four datasets, because the only reliable way to see what the numbers hide is to **look**.

---
### 📝 Summary
- Correlation r runs from −1 to 1. The sign is the direction, and |r| is the strength.
- `.corr()` for one pair, `df.corr()` for a matrix. Use the upper triangle to list unique pairs.
- **Spearman** ranks first: use it for ordinal data, curved-but-monotonic patterns, and outliers.
- r ≈ 0 rules out a **straight-line** relationship only.
- Correlation isn't causation. Look for confounders, and compare within groups.
- Identical statistics can hide completely different data (Anscombe). Always plot.

## 🏋️ Practice

### Practice 1 — Total and Its Parts (easy)
`Total` is the sum of the six stats. Compute the correlation of `Total` with each of the six stats, sorted from strongest to weakest. Which stat is most closely tied to `Total`?

In [18]:
# Your code here

stats = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

total_corr = (
    poke[stats + ['Total']]
    .corr()['Total']
    .drop('Total')
    .sort_values(ascending=False)
)

print(total_corr.round(3))


Sp. Atk    0.747
Attack     0.736
Sp. Def    0.718
HP         0.619
Defense    0.613
Speed      0.576
Name: Total, dtype: float64


In [20]:
print("Most strongly correlated with Total:")
print(total_corr.idxmax(), total_corr.max().round(3))

Most strongly correlated with Total:
Sp. Atk 0.747


### Practice 2 — Pearson vs. Spearman Table (medium)
Build a DataFrame with one row per pair of the six stats and two columns, `pearson` and `spearman` (hint: compute both matrices and stack each upper triangle). Which pair has the biggest difference between the two methods? What might cause that?

In [25]:
# Your code here

# Pearson correlations
pearson_corr = poke[stats].corr(method='pearson')

# Spearman correlations
spearman_corr = poke[stats].corr(method='spearman')

# Upper triangle mask
upper_mask_pearson = pearson_corr.where(np.triu(np.ones(pearson_corr.shape, dtype=bool), k=1))
upper_mask_spearman = spearman_corr.where(np.triu(np.ones(spearman_corr.shape, dtype=bool), k=1))

# One row per pair
pairs_pearson = upper_mask_pearson.stack().dropna().rename('r').sort_values(key=abs, ascending=False)
pairs_spearman = upper_mask_spearman.stack().dropna().rename('r').sort_values(key=abs, ascending=False)

pairs_pd = pd.DataFrame({
    'pearson': pairs_pearson,
    'spearman': pairs_spearman
})
pairs_pd.round(3)

pearson  spearman
Attack  Defense    0.439     0.515
        Sp. Atk    0.396     0.362
        Sp. Def    0.264     0.321
        Speed      0.381     0.373
Defense Sp. Atk    0.224     0.314
        Sp. Def    0.511     0.579
        Speed      0.015     0.093
HP      Attack     0.422     0.566
        Defense    0.240     0.433
        Sp. Atk    0.362     0.471
        Sp. Def    0.379     0.493
        Speed      0.176     0.266
Sp. Atk Sp. Def    0.506     0.572
        Speed      0.473     0.460
Sp. Def Speed      0.259     0.321

In [27]:
pairs['abs_diff'] = (
    pairs['pearson'] - pairs['spearman']
).abs()

print(
    pairs.sort_values('abs_diff', ascending=False)
         .round(3)
)

                 pearson  spearman  abs_diff
HP      Defense    0.240     0.433     0.194
        Attack     0.422     0.566     0.143
        Sp. Def    0.379     0.493     0.115
        Sp. Atk    0.362     0.471     0.108
Defense Sp. Atk    0.224     0.314     0.091
HP      Speed      0.176     0.266     0.090
Defense Speed      0.015     0.093     0.078
Attack  Defense    0.439     0.515     0.076
Defense Sp. Def    0.511     0.579     0.068
Sp. Atk Sp. Def    0.506     0.572     0.066
Sp. Def Speed      0.259     0.321     0.062
Attack  Sp. Def    0.264     0.321     0.057
        Sp. Atk    0.396     0.362     0.035
Sp. Atk Speed      0.473     0.460     0.013
Attack  Speed      0.381     0.373     0.008


*Your explanation here.*

The pair at the top of the table has the biggest difference between the two methods.

Pearson measures the strength of a linear relationship.
Spearman measures whether the variables move together in a monotonic (rank-order) way.

A large difference can happen when:

The relationship is curved rather than linear.
A few extreme Pokémon (outliers) pull the Pearson correlation up or down.
The variables tend to increase together in rank, but not at a constant rate.

### Practice 3 — Does the Correlation Hold Within Groups? (harder)
Compute r(`Attack`, `Defense`) for all Pokémon, then separately for Legendary and non-Legendary Pokémon, and then for each `Generation`. Put the results in a small table with the group size. Is the relationship consistent across groups?

In [29]:
# Your code here
rows = []

# All Pokémon
rows.append({
    'group': 'All',
    'n': len(poke),
    'r_attack_defense': poke['Attack'].corr(poke['Defense'])
})

# Legendary / non-Legendary
for legendary, grp in poke.groupby('Legendary'):
    rows.append({
        'group': f'Legendary={legendary}',
        'n': len(grp),
        'r_attack_defense': grp['Attack'].corr(grp['Defense'])
    })

# By Generation
for gen, grp in poke.groupby('Generation'):
    rows.append({
        'group': f'Generation {gen}',
        'n': len(grp),
        'r_attack_defense': grp['Attack'].corr(grp['Defense'])
    })

results = pd.DataFrame(rows)

print(results.round(3))

             group    n  r_attack_defense
0              All  800             0.439
1  Legendary=False  735             0.437
2   Legendary=True   65            -0.211
3     Generation 1  166             0.477
4     Generation 2  106             0.508
5     Generation 3  160             0.439
6     Generation 4  121             0.357
7     Generation 5  165             0.403
8     Generation 6   82             0.485


After running the code, compare the values in r_attack_defense:

If they're all fairly close (for example, around 0.4–0.6), then the relationship is consistent across groups.
If one generation or the Legendary group is much higher or lower, then the relationship varies by subgroup.
Differences among the smaller groups are more likely to be due to sampling variation because their n is smaller.

## ✏️ Your Turn
Compute the correlation matrix for `titanic`'s numeric columns (`numeric_only=True`). List the three strongest pairs. For **one** of them, write two sentences: one on what the correlation says, and one on a possible confounder or reason it might not be causal.

In [33]:
# Your turn here

corr = titanic.corr(numeric_only=True)

# Keep only the upper triangle
upper_titanic = corr.where(
    np.triu(np.ones(corr.shape, dtype=bool), k=1)
)
# One row per pair
pairs_titanic = (
    upper_titanic.stack()
    .rename('r')
    .sort_values(key=abs, ascending=False)
)

print(pairs_titanic.head(3).round(3))

Pclass  Fare    -0.549
SibSp   Parch    0.415
Pclass  Age     -0.369
Name: r, dtype: float64


*Your two sentences here.*

The correlation between Fare and Pclass indicates that passengers in higher classes generally paid higher fares. However, this does not necessarily mean class itself causes fare differences; fare is largely determined by ticket type, cabin location, route, and other factors that are closely tied to passenger class.

## 🔥 Challenge (Optional) — Spurious Correlations from Noise
Generate a DataFrame of 20 columns of **pure random noise**, 30 rows each (`np.random.normal(size=(30, 20))`). Compute its correlation matrix and find the strongest pair. How strong is it? What does that say about scanning a big table for the "best" correlation and reporting only that one?

In [ ]:
np.random.seed(21)
a = pd.Series(np.random.normal(50, 10, 30))
b = pd.Series(np.random.normal(50, 10, 30))      # unrelated to a

print('No outlier:   Pearson', round(a.corr(b), 3), '| Spearman', round(a.corr(b, method='spearman'), 3))

a_out = pd.concat([a, pd.Series([300])], ignore_index=True)
b_out = pd.concat([b, pd.Series([300])], ignore_index=True)
print('One outlier:  Pearson', round(a_out.corr(b_out), 3), '| Spearman', round(a_out.corr(b_out, method='spearman'), 3))

In [34]:
# Your code here

# Random noise: 30 rows, 20 columns
data_a = pd.DataFrame(
    np.random.normal(size=(30, 20)),
    columns=[f'col{i}' for i in range(20)]
)

# Correlation matrix
corr = data_a.corr()

# Upper triangle only
upper_a = corr.where(
    np.triu(np.ones(corr.shape, dtype=bool), k=1)
)

# One row per pair
pairs_a = (
    upper_a.stack()
    .rename('r')
    .sort_values(key=abs, ascending=False)
)

print(pairs_a.head())

col10  col11   -0.597365
col5   col11    0.468467
col8   col11   -0.448911
       col10    0.428660
col5   col6     0.409276
Name: r, dtype: float64


In [36]:
strongest_pair = pairs_a.index[0]
strongest_r = pairs_a.iloc[0]

print("Strongest pair:", strongest_pair)
print("Correlation:", round(strongest_r, 3))

Strongest pair: ('col10', 'col11')
Correlation: -0.597


The strongest correlation in a large table is not necessarily meaningful. When you scan many variables and report only the single largest correlation, you're effectively picking the winner of many random comparisons. Some correlations will appear strong purely by chance, even when no real relationship exists.

---
✅ Next up → [Describing Data Lab](06-Describing_Data_Lab.ipynb): put all five notebooks together on a dataset you haven't seen yet.